# 🔬 MorphoBricks Cellpose

Segments the images of a described MorphoBricks preprocessing run with **Cellpose 4.2.1.1** and writes the masks where preprocessing expects them. Your images are never changed: masks go to a separate masks folder, with a `segmentation_record.json` that says how they were made.

### What it does

Open a run → choose the output (cell or nuclei mask) and the channels **by channel ID** → choose the model and device → preview one example result → segment the whole experiment → rerun preprocessing with the masks folder.

### Before you start

- A finished preprocessing run (run `MorphoBricks_Preprocessing` first, with the experiment described). The run knows each image's axes and channel IDs; nothing is guessed here.
- A masks folder outside the experiment folder and the run's output folder.
- **Google Colab:** keep the run and the masks folder on Google Drive, under **/content/drive/MyDrive/**, and choose a GPU runtime (Runtime > Change runtime type).
- **JupyterLab / LabConstrictor:** use normal local paths.

### Things to know

- **2D images only for now.** An image with more than one Z plane or time point fails with a message that says so; it is never reduced to one plane.
- **Channels by ID.** You choose channels by their ID (for example `DNA`, `ACTIN`): a cell channel, a nuclear channel or both, and the output, a cell mask (cell channel, then the nuclear one if chosen) or a nuclei mask (nuclear channel only). In a run where some images have a channel that others lack, only the channels every image has can be used; the notebook shows the refusal as it is. Cellpose reads at most three channels and drops the rest by position, so more than three are refused.
- **Models:** `cpsam_v2` (default), `cpsam`, `cpdino`, `cpdino-vitb`, or a model file of your own. A name or file Cellpose cannot find is **refused**; Cellpose itself would silently use `cpsam_v2` instead.
- **Downloads:** built-in weights are downloaded from Hugging Face (mouseland/cellpose-sam) on first use. Nothing downloads unless you tick the download box in the form of cell 5. The `cpdino` models also need DINOv3 (installed only if you tick its box, at a pinned commit; see cell 5, which also says that the runtime must be restarted after that).
- **Device:** you choose `cuda`, `mps` or `cpu`. A GPU that is not available is **refused**; it never falls back to the CPU, and the notebook checks that every weight of the network sits on the device you chose (that is the network's placement only). Cellpose's own post-processing is not covered by that check: it may move its flow-threshold quality step to the CPU when GPU memory is short, and always computes the masks from the flows on the CPU on `mps`. That is Cellpose's own behaviour, it is **not refused**, and it is written to the record as a note.
- **Diameter:** in pixels. **Blank means the model's own native scale** (Cellpose 4.2.1.1 does not estimate a diameter). Zero is refused.
- **Normalisation** is always Cellpose's own, per image (each channel to its 1st-99th percentile); it is not a choice: images are never sent to Cellpose unnormalised.
- Every setting that ran (the full normalisation and the flow iterations included), the model's file name and its SHA-256, and whether the weights were downloaded in this session and from where, are written to the record.

### How to run

Run the code cells in order (click a cell and press **Shift + Enter**, or its ▶ button). From cell 3 on, each cell shows a form: fill it in and press its button, then go on to the next cell. You never edit code, in Google Colab, JupyterLab or LabConstrictor alike. When a step is refused, a red card under its button gives the reason; change the form and press the button again. After a restart of the runtime, run cell 1 again: it installs nothing that is already there.

___

## **1. Install Cellpose and MorphoBricks preprocessing**

Installs Cellpose 4.2.1.1 and the preprocessing commit this notebook is made for. Nothing is installed when it is already there, so running the cell again is safe. It prints **Ready** with the versions. It never installs DINOv3: that is a choice in cell 5.

**Google Colab:** if the installation upgrades `numpy` or `opencv` while the runtime already has them loaded, the cell stops and tells you to restart the runtime (Runtime > Restart session). Do that, then run cell 1 again: it installs nothing more and prints **Ready**.

In [ ]:
# @title 1. Install Cellpose and MorphoBricks preprocessing
import subprocess
import sys


class CellStop(RuntimeError):
    """Stop this cell with its message on one line, without a traceback (Colab and Jupyter)."""

    def _render_traceback_(self) -> list:
        return [str(self)]


if sys.version_info < (3, 11):
    raise CellStop("Select a Python 3.11 or newer kernel, then run this cell again.")

import json
from importlib import invalidate_caches
from importlib.metadata import PackageNotFoundError, distribution, version
from pathlib import Path

# LabConstrictor version identifiers.
current_version = "0.1.0"
notebook_name = "MorphoBricks_Cellpose"

# Both are pinned: Cellpose to the release this notebook is made for, preprocessing to the
# immutable source revision that holds this notebook's code.
CELLPOSE_VERSION = "4.2.1.1"
RELEASE_REF = "1fb5ac96861216b4269d5282e49d08397a35bf05"
REPOSITORY = "https://github.com/CellMigrationLab/morphobricks-preprocessing.git"
EXTRAS = "notebook,zarr" if "google.colab" in sys.modules else "notebook"

# DEVELOPMENT_OVERRIDE = True accepts whatever preprocessing is installed, as it is: another
# commit, an editable install or local folder, or one pip has no record of. Nothing replaces
# it, the imported copy is not checked against the install, and the banner then shows the
# commit pip recorded from REPOSITORY, or "commit unknown", and says the override is on; when
# none is installed, none is installed and the cell stops. Cellpose stays pinned. This cell
# runs no Git, so it cannot tell a clean checkout from an edited one (the Preprocessing
# notebook can). Set True only to develop against a local preprocessing.
DEVELOPMENT_OVERRIDE = False
SETUP_RECORD: dict = {}
# Packages pip may upgrade under a running kernel; each maps to the module that stays loaded.
LOADED_BY_KERNEL = {"numpy": "numpy", "opencv-python": "cv2", "opencv-python-headless": "cv2"}


def _version(name: str) -> str | None:
    """Return the installed version of a package, or None."""
    try:
        return version(name)
    except PackageNotFoundError:
        return None


def _direct_url(name: str) -> dict:
    """Return what pip recorded about where a package came from (PEP 610), or an empty dict.

    Empty when the package is not installed or its direct_url.json is missing or unreadable.
    """
    try:
        text = distribution(name).read_text("direct_url.json")
        found = json.loads(text) if text else {}
    except (PackageNotFoundError, OSError, ValueError):
        return {}
    return found if isinstance(found, dict) else {}


def _installed_commit(name: str) -> str | None:
    """Return the commit pip installed a package at from REPOSITORY with Git, else None.

    Only pip's record counts, never the version number: None for another URL, a local folder,
    an editable install or an archive, and for a missing or unreadable record.
    """
    found = _direct_url(name)
    info = found.get("vcs_info")
    if found.get("url") != REPOSITORY or not isinstance(info, dict) or info.get("vcs") != "git":
        return None
    commit = info.get("commit_id")
    return commit if isinstance(commit, str) else None


def _replace_other_commit(name: str, target: str) -> None:
    """Reinstall ``target`` over an installed ``name`` that pip did not record at RELEASE_REF.

    pip takes an installed package of the same version as already satisfied, even when asked
    for another commit, and installs nothing; so the pinned commit is forced. Only ``name`` is
    replaced: ``--no-deps`` never reinstalls numpy or another package the running kernel has
    loaded (the install after this one adds what is missing). pip restores the old copy if the
    new one cannot be installed. Nothing is done when ``name`` is not installed.
    """
    try:
        distribution(name)
    except PackageNotFoundError:
        return
    if _installed_commit(name) != RELEASE_REF:
        print(f"Replacing the installed {name} with commit {RELEASE_REF[:12]}...", flush=True)
        subprocess.run(
            [
                sys.executable,
                "-m",
                "pip",
                "install",
                "--disable-pip-version-check",
                "--force-reinstall",
                "--no-deps",
                target,
            ],
            check=True,
        )


def _loaded(name: str) -> bool:
    """Return whether this kernel already imported the package that ``name`` installs.

    A module imported before pip replaced its files stays the old copy until the kernel
    restarts. The import name is the distribution name with ``_`` for ``-``.
    """
    return name.replace("-", "_") in sys.modules


def _require_installed_copy(name: str, module_file: str) -> None:
    """Stop when this kernel imported ``name`` from outside the folder pip installed it in.

    pip's record describes only the installed files. A folder earlier on ``sys.path``, such as
    a checkout's ``src`` folder used as the working folder, is imported instead while the record
    still names the pinned commit.
    """
    installed = Path(distribution(name).locate_file("")).resolve()
    if not Path(module_file).resolve().is_relative_to(installed):
        raise CellStop(
            f"This kernel imports {name} from {module_file}, not from the copy pip installed in "
            f"{installed}. Open the notebook from a folder without another copy, restart the "
            "kernel, then run this cell again."
        )


def _install() -> None:
    """Install what is missing or differs from the pins, then report the versions."""
    wanted = []
    if _version("cellpose") != CELLPOSE_VERSION:
        wanted.append(f"cellpose=={CELLPOSE_VERSION}")
    preprocessing_ok = DEVELOPMENT_OVERRIDE or (
        _installed_commit("morphobricks-preprocessing") == RELEASE_REF
    )
    pinned = f"morphobricks-preprocessing[{EXTRAS}] @ git+{REPOSITORY}@{RELEASE_REF}"
    if not preprocessing_ok:
        wanted.append(pinned)
    if wanted:
        before = {name: _version(name) for name in LOADED_BY_KERNEL}
        print("Installing " + ", ".join(wanted) + "...", flush=True)
        try:
            if not preprocessing_ok:
                _replace_other_commit("morphobricks-preprocessing", pinned)
            subprocess.run(
                [sys.executable, "-m", "pip", "install", "--disable-pip-version-check", *wanted],
                check=True,
            )
        except (subprocess.CalledProcessError, OSError) as exc:
            raise CellStop(
                "The installation failed. Check the installation output and internet access, "
                "then run this cell again."
            ) from exc
        invalidate_caches()
        stale = [
            f"{name} {before[name]} -> {_version(name)}"
            for name, module in LOADED_BY_KERNEL.items()
            if before[name] and _version(name) != before[name] and module in sys.modules
        ]
        if not preprocessing_ok and _loaded("morphobricks-preprocessing"):
            # Same version, another commit: only the module, not the version, shows it.
            stale.append(f"MorphoBricks preprocessing to commit {RELEASE_REF[:12]}")
        if stale:
            raise CellStop(
                f"The installation changed {', '.join(stale)} while the runtime already has it "
                "loaded. Restart the runtime (Colab: Runtime > Restart session), then run this "
                "cell again: it installs nothing more."
            )
    if _version("cellpose") != CELLPOSE_VERSION:
        raise CellStop(
            f"Cellpose {CELLPOSE_VERSION} is not installed ({_version('cellpose')}). Restart "
            "the runtime and run this cell again."
        )
    commit = _installed_commit("morphobricks-preprocessing")
    if commit != RELEASE_REF and not DEVELOPMENT_OVERRIDE:
        # A restart would not help: pip kept the files (the cell forced the reinstall).
        raise CellStop(
            f"pip kept another MorphoBricks preprocessing: it recorded "
            f"{commit or 'no commit from ' + REPOSITORY}, not {RELEASE_REF}. Replace it with: "
            f'{sys.executable} -m pip install --force-reinstall --no-deps "{pinned}", then run '
            "this cell again."
        )
    try:
        from morphobricks_preprocessing import notebook_cellpose
    except ImportError as exc:
        cause = " ".join(f"{type(exc).__name__}: {exc}".split())
        raise CellStop(f"MorphoBricks preprocessing could not load ({cause}).") from exc
    if not DEVELOPMENT_OVERRIDE:
        _require_installed_copy("morphobricks-preprocessing", notebook_cellpose.__file__)
    SETUP_RECORD.clear()
    SETUP_RECORD.update(
        notebook_version=current_version,
        preprocessing_version=_version("morphobricks-preprocessing"),
        preprocessing_commit=commit,
        cellpose_version=_version("cellpose"),
        release_ref=RELEASE_REF,
        development_override=DEVELOPMENT_OVERRIDE,
    )
    override = (
        " DEVELOPMENT OVERRIDE is on: preprocessing is not the pinned commit."
        if DEVELOPMENT_OVERRIDE
        else ""
    )
    print(
        f"Ready: Cellpose {_version('cellpose')}; preprocessing "
        f"{_version('morphobricks-preprocessing')} (commit {commit or 'unknown'}).{override}\n"
        f"Notebook {current_version}.\n"
        "Next: run cell 2 (Mount Google Drive), then cell 3 (Open the run).",
        flush=True,
    )


_install()

___

## **2. Mount Google Drive**

**Google Colab:** connects your Google Drive. Allow access in the window that opens; your Drive is then the folder **/content/drive/MyDrive**. Keep the experiment and the output folder there: everything else on a Colab runtime is wiped when it disconnects.

**JupyterLab / LabConstrictor:** there is no Drive to mount. The cell only prints a note, and you use normal local paths in the app.

In [ ]:
# @title 2. Mount Google Drive
import sys

try:
    from morphobricks_preprocessing.notebook_stop import CellStop
except ImportError:
    raise ImportError("Run cell 1 first: it installs what this notebook needs.") from None

if "google.colab" in sys.modules:
    from google.colab import drive

    print("Google Colab: connecting Google Drive...", flush=True)
    try:
        drive.mount("/content/drive")
    except Exception as exc:
        # The cause stays on the one line: a stop must never hide what failed.
        cause = " ".join(f"{type(exc).__name__}: {exc}".split())
        raise CellStop(
            f"Google Drive could not be connected ({cause}). Allow Drive access, then run "
            "this cell again."
        ) from exc
    print("Google Drive is connected: your files are in /content/drive/MyDrive.", flush=True)
else:
    print("Not Google Colab: there is no Google Drive to mount here, so nothing was done.")

___

## **3. Open the preprocessing run**

Run the cell, then type or paste the run's output folder (it holds `run_manifest.json`) and the masks folder (outside the experiment and the run) in the form, or use **Browse…** where a folder dialog can open (not on Colab: there, copy the path from the **Files** panel). Press **Open the run**: the notebook opens the run with preprocessing's own checks and lists the channels with the number of images that have each. When the run cannot be opened (a run that was stopped, or images changed since the run), a red card gives the reason.

In [ ]:
# @title 3. Open the preprocessing run

try:
    from morphobricks_preprocessing.notebook_cellpose_form import CellposeNotebook
except ImportError:
    raise ImportError("Run cell 1 first: it installs what this notebook needs.") from None

workflow = CellposeNotebook()
workflow.open_step()

___

## **4. Choose the output and the channels**

Choose the **output**: a **Cell mask** or a **Nuclei mask**. The mask type is the name of the new masks (it becomes a folder in the masks folder and the name you declare in the experiment description); it follows the output (`Cells` or `Nuclei`), and you can type another. Press **Choose the channels**, then choose the channels **by channel ID** in the dropdowns that appear: a **cell channel** (cytoplasm or cell border) and a **nuclear channel**. Both are optional, but at least one is needed.

- **Cell mask:** Cellpose reads the cell channel, then the nuclear channel if you chose one. A cell channel is required.
- **Nuclei mask:** Cellpose reads the nuclear channel only. A nuclear channel is required.

The record names each channel's role and the output. A channel that some images lack says so, and choosing it shows why it is refused. If you change the output or the mask type afterwards, press **Choose the channels** again and load the model again.

In [ ]:
# @title 4. Choose the output and the channels

try:
    from morphobricks_preprocessing.notebook_stop import CellStop
except ImportError:
    raise ImportError("Run cell 1 first: it installs what this notebook needs.") from None

if "workflow" not in globals():
    raise CellStop("Run cell 3 first.")
workflow.channels_step()

___

## **5. Choose the model and the settings, then load the model**

Choose the model (or give a model file of your own), the device and the settings in the form, then press **Load the model**. Built-in weights are downloaded from **Hugging Face** on first use, and only if you tick the download box. When the form appears it says whether a GPU was found here, and which, before anything loads. The device is never chosen for you: a GPU that is not available is refused, not replaced by the CPU. The notebook loads the model, then checks that the weights file is the one you asked for and that the network's weights are on the device you asked for. It shows the file's SHA-256, which is written to the record. The diameter, the flow and cell probability thresholds and the batch size are under **Advanced Cellpose settings** (closed; open it to change them, the defaults are Cellpose's). The diameter is in pixels; blank is the model's native scale. Normalisation is not a setting: it is always Cellpose's own, per image (each channel to its 1st-99th percentile).

**The check covers every parameter and buffer of the network** (not Cellpose's own `device` attribute, which only repeats what it was given). Cellpose's post-processing is outside this check: it may move its flow-threshold quality step to the CPU when GPU memory is short, and always computes the masks from the flows on the CPU on `mps`. That is Cellpose's behaviour, not refused; the record carries it as a note, together with the full normalisation, the flow iterations, and whether the weights were downloaded now and from where.

**The `cpdino` models need DINOv3, and a restart.** Ticking the install box and pressing **Load the model** installs DINOv3 with `--no-deps` (so it cannot change your PyTorch) at commit `6876159a11b4`, **the head of its default branch on 22 November 2026, not verified against `cpdino`**. Cellpose imports DINOv3 only when it is first imported, so after installing DINOv3 **restart the runtime** (Colab: Runtime > Restart session), run cells 1 to 4 again and load a `cpdino` model; loading before the restart is refused with that instruction.

In [ ]:
# @title 5. Choose the model and the settings, then load the model

try:
    from morphobricks_preprocessing.notebook_stop import CellStop
except ImportError:
    raise ImportError("Run cell 1 first: it installs what this notebook needs.") from None

if "workflow" not in globals():
    raise CellStop("Run cell 3 first.")
workflow.model_step()

___

## **6. Preview one example result**

As in the ZeroCostDL4Mic Cellpose notebook: choose an image in the dropdown (every image of the run, starting on the first without a mask) and press **Preview**. The image is segmented with the loaded model and the settings of cell 5, and the notebook shows one panel per chosen channel, labelled with its channel ID and role (contrast stretched for display only; the masks and the stored values are never changed), then Cellpose's own panels: the input, the predicted outlines, the predicted masks and the predicted flows (cell pose), with the number of objects and the diameter used. **Nothing is written.**

Choose another image, or change the settings in the form of cell 5 (no need to load the model again), and press **Preview** again until it looks right. If you change the model, the model file or the device, press **Load the model** again first: the preview refuses until you do. The button is active once a model is loaded.

In [ ]:
# @title 6. Preview one example result

try:
    from morphobricks_preprocessing.notebook_stop import CellStop
except ImportError:
    raise ImportError("Run cell 1 first: it installs what this notebook needs.") from None

if "workflow" not in globals():
    raise CellStop("Run cell 3 first.")
workflow.preview_step()

___

## **7. Segment the whole experiment**

Press **Segment the whole experiment** (active once a model is loaded) to segment every image of the run that has no mask of this type yet, with the settings of cell 5 as they are now, and write each mask as a 2D integer image in the masks folder, then the record. A progress line shows how far it is. The table at the end lists **every** image as written, skipped (its mask matches its record), needs attention (a mask with no record entry or other bytes; not replaced) or failed with the error: nothing is skipped silently, and one failed image does not stop the others. Masks that already exist are replaced only if you tick **Replace masks that already exist** (off by default). If a mask's record entry cannot be written, that mask is rolled back (an earlier mask restored) and the batch stops.

In [ ]:
# @title 7. Segment the whole experiment

try:
    from morphobricks_preprocessing.notebook_stop import CellStop
except ImportError:
    raise ImportError("Run cell 1 first: it installs what this notebook needs.") from None

if "workflow" not in globals():
    raise CellStop("Run cell 3 first.")
workflow.segment_step()

___

## **8. What to do next**

Press **Show the next steps** for these steps with your mask type and masks folder:

1. Declare the new mask type in the experiment description (preprocessing notebook, step 4, or `morphobricks.toml`): for example `[masks.Cells]` with `source_type = "Cells"` and `representation = "instances"`.
2. Save the description.
3. Run preprocessing again with the masks folder (the histograms are reused).
4. Describe the masks.
5. Then continue with MorphoFeature.

In [ ]:
# @title 8. What to do next

try:
    from morphobricks_preprocessing.notebook_stop import CellStop
except ImportError:
    raise ImportError("Run cell 1 first: it installs what this notebook needs.") from None

if "workflow" not in globals():
    raise CellStop("Run cell 3 first.")
workflow.next_step()

___

## **Versions and citation**

**Versions.** This notebook runs **Cellpose 4.2.1.1** (pinned in cell 1, which also pins the MorphoBricks preprocessing commit and prints both versions when it says **Ready**). Each batch in `<masks>/<mask type>/segmentation_record.json` stores the tool and its exact version (`tool`, `tool_version`; again as `cellpose_version` in the batch's `parameters`), the model's name, weights file and SHA-256, the DINOv3 commit for a `cpdino` model (`dinov3_commit`), whether the weights were downloaded and from where, the device and every setting that ran. The preprocessing version is not in that record: cell 1 prints it, and the run's own `run_manifest.json` records the preprocessing that made the run.

**Citation.** As Cellpose's own README (release 4.2.1.1) asks:

- If you use Cellpose-SAM (the `cpsam_v2` and `cpsam` models), cite the Cellpose-SAM paper: Pachitariu, M., Rariden, M., & Stringer, C. (2025). Cellpose-SAM: superhuman generalization for cellular segmentation. *bioRxiv*. https://www.biorxiv.org/content/10.1101/2025.04.28.651001v1
- If you use Cellpose 1, 2 or 3, cite the Cellpose 1.0 paper: Stringer, C., Wang, T., Michaelos, M., & Pachitariu, M. (2021). Cellpose: a generalist algorithm for cellular segmentation. *Nature Methods*, 18(1), 100-106.
- If you use human-in-the-loop training, also cite the Cellpose 2.0 paper: Pachitariu, M. & Stringer, C. (2022). Cellpose 2.0: how to train your own model. *Nature Methods*, 1-8.

TODO: neither the README nor the model documentation of Cellpose 4.2.1.1 names a paper for the `cpdino` models (CellposeDINO, June 2026, DINOv3 backbone). Check Cellpose's current documentation for their citation before publishing results made with them.